In [1]:
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from helper_code import find_records, get_age, get_sex, load_header, get_label, load_signals

In [2]:
%load_ext autoreload

%autoreload 2

SEED = 42
BATCH_SIZE = 64

In [3]:
def get_features_and_labels(records_id: str, data_folder: Path):
    record_path = (data_folder / records_id).absolute().__str__()
    header = load_header(record_path)
    signals, fields = load_signals(record_path)
    label = get_label(header)
    age = get_age(header)
    sex = get_sex(header)
    return signals, label, age, sex

In [4]:
class ECGDataset(Dataset):
    def __init__(self, features, labels, max_len: int=4096):
        self.features = features
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        signals = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if signals.size(0) < self.max_len:
            signals = torch.cat([signals, torch.zeros(self.max_len - signals.size(0), signals.size(1))])

        return signals, label

In [5]:
train_data_folder = Path('code15_wfdb')

In [6]:
train_records = find_records(train_data_folder.absolute().__str__())

In [7]:
data = Parallel(n_jobs=-1)(delayed(get_features_and_labels)(record_id, train_data_folder) for record_id in tqdm(train_records, desc='Loading signals'))

Loading signals: 100%|██████████| 13092/13092 [00:21<00:00, 602.55it/s]


In [8]:
features, labels, ages, sexes = zip(*data)

In [9]:
X, y = features, labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED)

train_dataset = ECGDataset(X_train, y_train)
val_dataset = ECGDataset(X_val, y_val)
test_dataset = ECGDataset(X_test, y_test)
train_dloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dloader = DataLoader(val_dataset, shuffle=False)
test_dloader = DataLoader(test_dataset, shuffle=False)

In [10]:
def _padding(downsample, kernel_size):
    """Compute required padding"""
    padding = max(0, int(np.floor((kernel_size - downsample + 1) / 2)))
    return padding


def _downsample(n_samples_in, n_samples_out):
    """Compute downsample rate"""
    downsample = int(n_samples_in // n_samples_out)
    if downsample < 1:
        raise ValueError("Number of samples should always decrease")
    if n_samples_in % n_samples_out != 0:
        raise ValueError("Number of samples for two consecutive blocks "
                         "should always decrease by an integer factor.")
    return downsample

class ResBlock1d(nn.Module):
    """Residual network unit for unidimensional signals."""

    def __init__(self, n_filters_in, n_filters_out, downsample, kernel_size, dropout_rate):
        if kernel_size % 2 == 0:
            raise ValueError("The current implementation only support odd values for `kernel_size`.")
        super(ResBlock1d, self).__init__()
        # Forward path
        padding = _padding(1, kernel_size)
        self.conv1 = nn.Conv1d(n_filters_in, n_filters_out, kernel_size, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm1d(n_filters_out)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        padding = _padding(downsample, kernel_size)
        self.conv2 = nn.Conv1d(n_filters_out, n_filters_out, kernel_size,
                               stride=downsample, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm1d(n_filters_out)
        self.dropout2 = nn.Dropout(dropout_rate)

        # Skip connection
        skip_connection_layers = []
        # Deal with downsampling
        if downsample > 1:
            maxpool = nn.MaxPool1d(downsample, stride=downsample)
            skip_connection_layers += [maxpool]
        # Deal with n_filters dimension increase
        if n_filters_in != n_filters_out:
            conv1x1 = nn.Conv1d(n_filters_in, n_filters_out, 1, bias=False)
            skip_connection_layers += [conv1x1]
        # Build skip conection layer
        if skip_connection_layers:
            self.skip_connection = nn.Sequential(*skip_connection_layers)
        else:
            self.skip_connection = None

    def forward(self, x, y):
        """Residual unit."""
        if self.skip_connection is not None:
            y = self.skip_connection(y)
        else:
            y = y
        # 1st layer
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)

        # 2nd layer
        x = self.conv2(x)
        x += y  # Sum skip connection and main connection
        y = x
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)
        return x, y

class ResNet1d(torch.nn.Module):
    def __init__(self, input_dim: int, blocks_dim: int, n_classes: int, kernel_size: int, dropout_rate: float = 0.2):
        super().__init__()

        n_filters_in, n_filters_out = input_dim[0], blocks_dim[0][0]
        n_samples_in, n_samples_out = input_dim[1], blocks_dim[0][1]
        downsample = _downsample(n_samples_in, n_samples_out)
        padding = _padding(downsample, kernel_size)
        self.conv1 = nn.Conv1d(n_filters_in, n_filters_out, kernel_size=kernel_size, stride=downsample, padding=padding)
        self.bn1 = nn.BatchNorm1d(n_filters_out)
    
        self.res_blocks = []
        for i, (n_filters, n_samples) in enumerate(blocks_dim):
            n_filters_in, n_filters_out = n_filters_out, n_filters
            n_samples_in, n_samples_out = n_samples_out, n_samples
            downsample = _downsample(n_samples_in, n_samples_out)
            resblk1d = ResBlock1d(n_filters_in, n_filters_out, downsample, kernel_size, dropout_rate)
            self.add_module('resblock1d_{0}'.format(i), resblk1d)
            self.res_blocks += [resblk1d]

        # Linear layer
        n_filters_last, n_samples_last = blocks_dim[-1]
        last_layer_dim = n_filters_last * n_samples_last
        self.lin = nn.Linear(last_layer_dim, n_classes)
        self.n_blk = len(blocks_dim)
        
    
    def forward(self, x):
        """Implement ResNet1d forward propagation"""
        # First layers
        x = self.conv1(x)
        x = self.bn1(x)

        # Residual blocks
        y = x
        for blk in self.res_blocks:
            x, y = blk(x, y)

        # Flatten array
        x = x.view(x.size(0), -1)

        # Fully conected layer
        x = self.lin(x)
        return x

In [11]:
with torch.no_grad():
    torch.cuda.empty_cache()

In [12]:
N_LEADS = 12  # the 12 leads
N_CLASSES = 1  # just the Chagas disease
model = ResNet1d(input_dim=(N_LEADS, 4096),
                     blocks_dim=list(zip([8, 16], [4096, 16])),
                     n_classes=N_CLASSES,
                     kernel_size=17,
                     dropout_rate=0.2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

ResNet1d(
  (conv1): Conv1d(12, 8, kernel_size=(17,), stride=(1,), padding=(8,))
  (bn1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (resblock1d_0): ResBlock1d(
    (conv1): Conv1d(8, 8, kernel_size=(17,), stride=(1,), padding=(8,), bias=False)
    (bn1): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (dropout1): Dropout(p=0.2, inplace=False)
    (conv2): Conv1d(8, 8, kernel_size=(17,), stride=(1,), padding=(8,), bias=False)
    (bn2): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (dropout2): Dropout(p=0.2, inplace=False)
  )
  (resblock1d_1): ResBlock1d(
    (conv1): Conv1d(8, 16, kernel_size=(17,), stride=(1,), padding=(8,), bias=False)
    (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (dropout1): Dropout(p=0.2, inplace=False)
    (conv2): Conv1d(16, 16, kernel_size=(17,), stride=(256,), bias=

In [13]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [14]:
import pandas as pd


tqdm.write("Training...")
start_epoch = 0
best_loss = np.Inf
history = pd.DataFrame(columns=['epoch', 'train_loss', 'valid_loss', 'lr', 'f1', 'accuracy'])

for epoch in range(start_epoch, 100):
    model.train()
    total_loss = 0
    n_entries = 0
    train_desc = "Epoch {:2d}: train - Loss: {:.6f}"
    train_bar = tqdm(initial=0, leave=True, total=len(train_dloader), desc=train_desc.format(epoch, 0))
    for signals, labels in train_dloader:
        signals = signals.transpose(1, 2)
        signals, labels = signals.to(device), labels.to(device)

        model.zero_grad()
        optimizer.zero_grad()

        outputs = model(signals)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.detach().cpu().numpy()
        n_entries += len(signals)
        train_bar.desc = train_desc.format(epoch, total_loss / n_entries)
        train_bar.update(1)
train_bar.close()

Training...


Epoch  0: train - Loss: 0.000000:   0%|          | 0/246 [00:00<?, ?it/s]../aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [1,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [2,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [3,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [4,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:240: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [7,0,0] Assertion `t >= 0 && t 

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
